# LeetCode 3491 — Phone Number Prefix bằng Python

Link đề: https://leetcode.com/problems/phone-number-prefix/ · Độ khó: **Easy** · Chủ đề: Array, String, Sorting, Trie

> **Bài Premium:** đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium. (Tên bài, chữ ký hàm, hai ví dụ và constraints bên dưới đã đối chiếu qua nguồn công khai; riêng việc xử lý các số **trùng hệt nhau** trong `numbers` cần đối chiếu đề thật — notebook này coi hai chuỗi bằng nhau là "một số là tiền tố của số kia" và không dùng input trùng trong test.)

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def phonePrefix(self, numbers: List[str]) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu khái niệm **tiền tố (prefix)** của chuỗi và phân biệt "bắt đầu bằng" với "nằm trong" (`startswith` khác `in`).
2. Dùng `str.startswith`, cắt lát `s[:L]`, `sorted` trên chuỗi, và `set` để kiểm tra tiền tố.
3. Hiểu vì sao **sắp xếp từ điển** đưa mọi cặp (tiền tố, số dài hơn) về **sát nhau**, nên chỉ cần so sánh các cặp kề nhau.
4. Viết 4 cách: so mọi cặp, sort + so cặp kề, `set` các tiền tố, và **Trie** (cây tiền tố) tự cài đặt bằng `dict`.
5. Tránh các lỗi: dùng `in` thay `startswith`, quên sort, sort theo **giá trị số** (làm mất số `0` đứng đầu), và chỉ kiểm tra một chiều của cặp.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `numbers`, mỗi chuỗi là một số điện thoại (chỉ gồm chữ số `0`–`9`, có thể bắt đầu bằng `0`). Trả về `true` nếu **không** có số nào là **tiền tố** của một số khác; ngược lại trả về `false`.

Chuỗi `a` là tiền tố của `b` nếu `b` bắt đầu bằng `a` (ví dụ `"001"` là tiền tố của `"00153"`).

| Ví dụ | numbers | Output | Giải thích |
|---|---|---|---|
| 1 | `["1","2","4","3"]` | `true` | không số nào là tiền tố của số khác |
| 2 | `["001","007","15","00153"]` | `false` | `"001"` là tiền tố của `"00153"` |

Constraints: `2 <= numbers.length <= 50`; `1 <= numbers[i].length <= 50`; `numbers[i]` chỉ gồm các chữ số `0`–`9`.

Chú ý: đây là **chuỗi**, không phải số nguyên. `"001"` và `"1"` là hai số khác nhau; so sánh theo **ký tự**, không theo giá trị.

In [1]:
examples = [(["1", "2", "4", "3"], True), (["001", "007", "15", "00153"], False)]
for numbers, expected in examples:
    pairs = [(a, b) for a in numbers for b in numbers if a is not b and b.startswith(a)]
    print(numbers, "-> các cặp (tiền tố, số dài hơn):", pairs, "-> kết quả", not pairs, "(mong đợi", expected, ")")

['1', '2', '4', '3'] -> các cặp (tiền tố, số dài hơn): [] -> kết quả True (mong đợi True )
['001', '007', '15', '00153'] -> các cặp (tiền tố, số dài hơn): [('001', '00153')] -> kết quả False (mong đợi False )


## 3. Kiến thức nền

### 3.1. Tiền tố, `startswith` và cắt lát

`b.startswith(a)` là `True` khi `b` bắt đầu bằng `a`; tương đương `b[:len(a)] == a`. Nếu `a` dài hơn `b` thì luôn `False`. Hai chuỗi **bằng nhau** cũng thỏa `startswith` (mỗi chuỗi là tiền tố của chính nó).

In [2]:
print("00153".startswith("001"), "00153".startswith("15"), "15".startswith("153"))
print("00153"[:3] == "001")
print("abc".startswith("abc"))        # bằng nhau cũng tính là bắt đầu bằng

True False False
True
True


### 3.2. `startswith` khác `in`

`a in b` hỏi "`a` có xuất hiện **ở đâu đó** trong `b` không", chưa chắc ở đầu. `"12" in "312"` là `True` nhưng `"312".startswith("12")` là `False`. Bài này cần `startswith`.

In [3]:
print("12" in "312", "312".startswith("12"))
print("15" in "00153", "00153".startswith("15"))

True False
True False


### 3.3. Sắp xếp chuỗi theo thứ tự từ điển

`sorted(list_of_str)` so sánh từng ký tự từ trái sang phải; nếu một chuỗi là tiền tố của chuỗi kia thì chuỗi ngắn hơn đứng **trước**. Đây khác với sắp xếp theo giá trị số: `sorted(["9", "10"])` ra `["10", "9"]` còn `sorted([9, 10])` ra `[9, 10]`.

In [4]:
print(sorted(["9", "10", "1"]))
print(sorted([9, 10, 1]))
print(sorted(["001", "007", "15", "00153"]))

['1', '10', '9']
[1, 9, 10]
['001', '00153', '007', '15']


### 3.4. `set` — tra cứu `O(1)` trung bình

`x in some_set` rất nhanh vì `set` dùng bảng băm. Ta dùng nó để hỏi "tiền tố này có nằm trong tập các số điện thoại không?" cho từng tiền tố của một số.

In [5]:
S = {"001", "007", "15", "00153"}
for p in ["0", "00", "001", "0015", "00153"]:
    print(repr(p), p in S)

'0' False
'00' False
'001' True
'0015' False
'00153' True


### 3.5. Trie (cây tiền tố) bằng `dict` lồng nhau

Trie là cây mà mỗi cạnh mang một ký tự; đường đi từ gốc xuống một nút là một tiền tố. Ta lưu mỗi nút là một `dict` ánh xạ ký tự -> nút con, và đánh dấu nút kết thúc của một số bằng khóa đặc biệt (ví dụ `"$"`). Chèn một chuỗi mất `O(độ dài)`.

In [6]:
root = {}
for word in ["001", "007", "15"]:
    node = root
    for ch in word:
        node = node.setdefault(ch, {})   # có rồi thì lấy, chưa có thì tạo nút mới
    node["$"] = True                      # đánh dấu kết thúc số
print(root)

{'0': {'0': {'1': {'$': True}, '7': {'$': True}}}, '1': {'5': {'$': True}}}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tay làm với `["001","007","15","00153"]`: ta cần tìm một cặp `(a, b)` với `b` bắt đầu bằng `a`.

```
"001"   là tiền tố của "00153"?  "00153"[:3] = "001"  -> CÓ  => trả về false
```

Cách tự nhiên nhất là thử **mọi cặp**. Nhưng ta nhận xét thêm: nếu `a` là tiền tố của `b` thì `a` và `b` ở **gần nhau** khi sắp xếp từ điển (mọi chuỗi nằm giữa chúng cũng bắt đầu bằng `a`). Sắp xếp xong:

```
["001", "00153", "007", "15"]
   ^______^   -> hai chuỗi kề nhau, "00153" bắt đầu bằng "001"
```

Do đó chỉ cần nhìn **các cặp kề nhau**. Đó là bước đột phá cho Cách 2.

## 5. Cách 1 — Brute Force / baseline

Thử mọi cặp có thứ tự `(i, j)` với `i != j`: nếu `numbers[j].startswith(numbers[i])` thì có một số là tiền tố của số khác -> `False`. Hết mà không thấy -> `True`.

Nút thắt: `n` số có `n(n-1)` cặp, mỗi lần `startswith` mất tới `O(m)` (`m` là độ dài lớn nhất).

In [7]:
class SolutionBrute:
    def phonePrefix(self, numbers):
        n = len(numbers)
        for i in range(n):
            for j in range(n):
                if i != j and numbers[j].startswith(numbers[i]):
                    return False
        return True

print(SolutionBrute().phonePrefix(["1", "2", "4", "3"]))
print(SolutionBrute().phonePrefix(["001", "007", "15", "00153"]))

True
False


**Độ phức tạp:** `O(n² · m)` thời gian, `O(1)` bộ nhớ thêm. Với `n <= 50`, `m <= 50` là `50·49·50 ≈ 122 500` thao tác nên rất nhanh; nhưng ta vẫn học các cách tốt hơn vì chúng là nền cho các bài tiền tố lớn hơn.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề (sort):** nếu `a` là tiền tố **thực sự** của `b` (`a != b`), thì trong thứ tự từ điển, `a` đứng ngay trước **một** chuỗi cũng bắt đầu bằng `a`.

Chứng minh ngắn: gọi `c` là chuỗi đứng ngay sau `a`. Ta có `a < c <= b` (vì `a < b`), mà mọi chuỗi `c` với `a <= c <= b` cũng phải bắt đầu bằng `a` (nếu `c` khác `a` ở ký tự thứ `t <= len(a)` thì hoặc `c < a` hoặc `c > b`, vì `a` và `b` giống nhau trên `len(a)` ký tự đầu). Vậy `c` bắt đầu bằng `a`.

**Quy tắc "nếu… thì…":**

- Sort `numbers`. Nếu tồn tại `i` sao cho `sorted[i+1].startswith(sorted[i])` -> `False`.
- Ngược lại -> `True`.

**Ý tưởng thứ hai (set):** một số `x` có `len(x) - 1` tiền tố thực sự; kiểm tra từng tiền tố có nằm trong `set(numbers)` không.

**Ý tưởng thứ ba (Trie):** chèn từng số; nếu trong lúc đi xuống gặp một nút đánh dấu "kết thúc số" -> có số là tiền tố của số đang chèn; nếu chèn xong mà nút hiện tại **đã có con** -> số đang chèn là tiền tố của số đã chèn trước.

In [8]:
numbers = ["001", "007", "15", "00153"]
s = sorted(numbers)
print("sau sort:", s)
for i in range(len(s) - 1):
    print(f"{s[i]!r} -> {s[i+1]!r}: {s[i+1].startswith(s[i])}")

sau sort: ['001', '00153', '007', '15']
'001' -> '00153': True
'00153' -> '007': False
'007' -> '15': False


## 7. Các cách tiếp theo

Ta có 4 cách: (1) brute so mọi cặp; (2) sort + cặp kề; (3) `set` tiền tố; (4) Trie.

### Cách 2 — Sort + so cặp kề nhau

Sort `numbers`, rồi chỉ so `s[i+1].startswith(s[i])`. Chi phí chủ yếu là sort `O(n log n)` so sánh chuỗi (mỗi so sánh `O(m)`).

In [9]:
class SolutionSort:
    def phonePrefix(self, numbers):
        s = sorted(numbers)
        for i in range(len(s) - 1):
            if s[i + 1].startswith(s[i]):
                return False
        return True

print(SolutionSort().phonePrefix(["1", "2", "4", "3"]))
print(SolutionSort().phonePrefix(["001", "007", "15", "00153"]))

True
False


Độ phức tạp Cách 2: `O(n · m · log n)` thời gian (sort) + `O(n · m)` cho vòng so cặp kề; bộ nhớ `O(n · m)` cho bản sao đã sắp xếp (`sorted` tạo danh sách mới).

### Cách 3 — `set` các số, kiểm tra từng tiền tố

Tạo `S = set(numbers)`. Với mỗi số `x` và mỗi độ dài `L` từ `1` đến `len(x) - 1`, nếu `x[:L] in S` thì `x[:L]` là một số điện thoại là tiền tố của `x` -> `False`. Thêm một điều kiện: nếu có số **trùng hệt** (`len(S) != len(numbers)`) thì theo quy ước của notebook cũng trả `False`.

In [10]:
class SolutionSet:
    def phonePrefix(self, numbers):
        S = set(numbers)
        if len(S) != len(numbers):          # quy ước: hai số bằng nhau thì số này là tiền tố của số kia
            return False
        for x in numbers:
            for L in range(1, len(x)):
                if x[:L] in S:
                    return False
        return True

print(SolutionSet().phonePrefix(["1", "2", "4", "3"]))
print(SolutionSet().phonePrefix(["001", "007", "15", "00153"]))

True
False


Độ phức tạp Cách 3: mỗi số có `m` tiền tố, mỗi lần cắt lát và băm tốn `O(m)` -> `O(n · m²)` thời gian, `O(n · m)` bộ nhớ cho `set`. Với `m <= 50` vẫn ổn.

### Cách 4 — Trie

Chèn từng số vào Trie. Tại mỗi ký tự: nếu nút hiện tại đã có khóa `"$"` (kết thúc của một số đã chèn) thì số đã chèn là tiền tố của số đang chèn. Sau khi chèn hết ký tự: nếu nút hiện tại đã có con (kể cả `"$"` nếu trùng hệt) thì số đang chèn là tiền tố của một số đã chèn.

In [11]:
class SolutionTrie:
    def phonePrefix(self, numbers):
        root = {}
        for word in numbers:
            node = root
            for ch in word:
                if "$" in node:                 # một số đã chèn là tiền tố của word
                    return False
                node = node.setdefault(ch, {})
            if node:                            # word là tiền tố của một số đã chèn (hoặc trùng hệt)
                return False
            node["$"] = True
        return True

print(SolutionTrie().phonePrefix(["1", "2", "4", "3"]))
print(SolutionTrie().phonePrefix(["001", "007", "15", "00153"]))

True
False


Độ phức tạp Cách 4: mỗi ký tự được xử lý một lần -> `O(S)` với `S` là **tổng độ dài** các số (`S <= n·m`), bộ nhớ `O(S)` cho cây. Đây là cách tốt nhất về lý thuyết và là mẫu chuẩn cho các bài "tập hợp chuỗi + tiền tố".

## 8. TRACE từng bước

Trace Cách 4 (Trie) với `numbers = ["001", "007", "15", "00153"]`:

| bước | số đang chèn | cây sau khi chèn | phát hiện |
|---|---|---|---|
| 1 | `"001"` | `0 -> 0 -> 1 -> $` | không |
| 2 | `"007"` | thêm `0 -> 0 -> 7 -> $` (dùng chung `0 -> 0`) | không |
| 3 | `"15"` | thêm `1 -> 5 -> $` | không |
| 4 | `"00153"` | đi `0`, `0`, `1`: tới nút `1` có `"$"` | **có**: `"001"` là tiền tố -> `False` |

In [12]:
def trace_trie(numbers):
    root = {}
    for step, word in enumerate(numbers, start=1):
        node = root
        print(f"bước {step}: chèn {word!r}")
        for pos, ch in enumerate(word):
            if "$" in node:
                print(f"   tại vị trí {pos}: nút hiện tại có '$' -> một số đã chèn là tiền tố của {word!r} => False")
                return False
            node = node.setdefault(ch, {})
            print(f"   đi theo {ch!r}, nút con có các khóa {sorted(node)}")
        if node:
            print(f"   {word!r} đã là tiền tố của số đã chèn (nút có con {sorted(node)}) => False")
            return False
        node["$"] = True
    print("không có tiền tố nào => True")
    return True

trace_trie(["001", "007", "15", "00153"])
trace_trie(["1", "2", "4", "3"])
trace_trie(["153", "15"])

bước 1: chèn '001'
   đi theo '0', nút con có các khóa []
   đi theo '0', nút con có các khóa []
   đi theo '1', nút con có các khóa []
bước 2: chèn '007'
   đi theo '0', nút con có các khóa ['0']
   đi theo '0', nút con có các khóa ['1']
   đi theo '7', nút con có các khóa []
bước 3: chèn '15'
   đi theo '1', nút con có các khóa []
   đi theo '5', nút con có các khóa []
bước 4: chèn '00153'
   đi theo '0', nút con có các khóa ['0']
   đi theo '0', nút con có các khóa ['1', '7']
   đi theo '1', nút con có các khóa ['$']
   tại vị trí 3: nút hiện tại có '$' -> một số đã chèn là tiền tố của '00153' => False
bước 1: chèn '1'
   đi theo '1', nút con có các khóa []
bước 2: chèn '2'
   đi theo '2', nút con có các khóa []
bước 3: chèn '4'
   đi theo '4', nút con có các khóa []
bước 4: chèn '3'
   đi theo '3', nút con có các khóa []
không có tiền tố nào => True
bước 1: chèn '153'
   đi theo '1', nút con có các khóa []
   đi theo '5', nút con có các khóa []
   đi theo '3', nút con có các khóa [

False

## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def phonePrefix(self, numbers: List[str]) -> bool:
        s = sorted(numbers)
        for i in range(len(s) - 1):
            if s[i + 1].startswith(s[i]):
                return False
        return True

print(Solution().phonePrefix(["1", "2", "4", "3"]), Solution().phonePrefix(["001", "007", "15", "00153"]))

True False


## 10. Vì sao thuật toán đúng

- **Cách 1:** thử mọi cặp có thứ tự nên không bỏ sót; trả `False` đúng khi tồn tại cặp `(a, b)`, `a != b` (vị trí), `b` bắt đầu bằng `a`.
- **Cách 2:** (một chiều) nếu tìm thấy cặp kề thỏa -> đúng là có tiền tố. (chiều ngược) nếu tồn tại một số là tiền tố của số khác thì theo **bổ đề** ở mục 6 sẽ có một cặp **kề nhau** sau khi sort thỏa `startswith` -> thuật toán bắt được. Khi hai số bằng nhau chúng kề nhau (sort ổn định) nên cũng bị bắt, khớp với quy ước.
- **Cách 3:** mọi tiền tố thực sự của `x` đều có độ dài từ `1` đến `len(x)-1`; ta thử hết; trùng hệt xử lý riêng bằng kích thước `set`.
- **Cách 4 (Trie):** đường đi từ gốc là các tiền tố. Khi chèn `w`: gặp nút kết thúc giữa đường -> một số cũ là tiền tố của `w`; kết thúc tại nút có con -> `w` là tiền tố của một số cũ. Hai khả năng này bao phủ mọi quan hệ tiền tố giữa `w` và các số chèn trước; vì quan hệ là đối xứng về thứ tự chèn (mỗi cặp có một số chèn sau) nên mọi cặp đều bị kiểm tra.
- **Bộ test** so khớp cả bốn cách với một oracle viết bằng cắt lát trên hàng nghìn input ngẫu nhiên có nhiều tiền tố.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số chuỗi, `m` là độ dài lớn nhất, `S` tổng độ dài.

| Cách | Time | Space |
|---|---|---|
| 1. So mọi cặp | `O(n² · m)` | `O(1)` |
| 2. Sort + cặp kề | `O(n · m · log n)` | `O(n · m)` |
| 3. `set` + mọi tiền tố | `O(n · m²)` | `O(n · m)` |
| 4. Trie | `O(S)` ≤ `O(n · m)` | `O(S)` |

Với giới hạn `n, m <= 50` mọi cách đều tức thì. Trong phỏng vấn: nêu Cách 2 (ngắn, đẹp) trước, nhắc Cách 4 (Trie) như tối ưu về lý thuyết.

## 12. Bộ test

Test 1: ví dụ đề và các edge case. Mọi input tuân thủ `2 <= n <= 50`, mỗi chuỗi dài `1..50`, chỉ gồm chữ số. Các input **không** có chuỗi trùng hệt (xem ghi chú Premium ở đầu notebook).

In [14]:
impls = [SolutionBrute(), SolutionSort(), SolutionSet(), SolutionTrie(), Solution()]
tests = [
    (["1", "2", "4", "3"], True),
    (["001", "007", "15", "00153"], False),
    (["1", "12"], False),                         # n = 2, tiền tố một ký tự
    (["12", "1"], False),                         # tiền tố đứng sau
    (["1", "2"], True),                           # n = 2, không liên quan
    (["12", "312"], True),                        # "12" nằm trong "312" nhưng không phải tiền tố
    (["0", "00"], False),                         # toàn số 0
    (["01", "1"], True),                          # "01" và "1": giá trị bằng nhau nhưng không phải tiền tố
    (["123", "124", "12"], False),                # tiền tố nằm giữa
    (["9" * 50, "9" * 49], False),                # độ dài lớn nhất 50
    (["9" * 50, "8" + "9" * 49], True),           # cùng độ dài, khác ký tự đầu
    (["1", "2", "3", "4", "5", "6", "7", "8", "9", "0"], True),   # mười số một chữ số
    (["10", "100", "1000", "10000"], False),      # chuỗi các tiền tố lồng nhau
]
for numbers, exp in tests:
    assert 2 <= len(numbers) <= 50 and len(set(numbers)) == len(numbers)
    assert all(1 <= len(x) <= 50 and x.isdigit() for x in numbers)
    for im in impls:
        got = im.phonePrefix(list(numbers))
        assert got is exp, (type(im).__name__, numbers, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 13


Test 2: ngẫu nhiên có seed. Alphabet `01` để các tiền tố xuất hiện thường xuyên, alphabet đủ `0-9` và độ dài lớn để có cả trường hợp `True`. Oracle độc lập: so từng cặp bằng cắt lát `b[:len(a)] == a` (không dùng `startswith`).

In [15]:
import random
random.seed(3491)

def oracle(numbers):
    for i, a in enumerate(numbers):
        for j, b in enumerate(numbers):
            if i != j and len(a) <= len(b) and b[:len(a)] == a:
                return False
    return True

trues = falses = 0
for t in range(3000):
    alphabet = random.choice(["01", "0123456789", "012"])
    max_len = random.choice([3, 6, 10, 50])
    n = random.randint(2, 50)
    pool = set()
    for _ in range(n):
        L = random.randint(1, max_len)
        pool.add("".join(random.choice(alphabet) for _ in range(L)))
    numbers = list(pool)
    if len(numbers) < 2:
        continue
    random.shuffle(numbers)
    assert 2 <= len(numbers) <= 50 and all(1 <= len(x) <= 50 and x.isdigit() for x in numbers)
    ref = oracle(numbers)
    trues += ref
    falses += (not ref)
    for im in impls:
        assert im.phonePrefix(list(numbers)) is ref, (type(im).__name__, numbers)
print("test ngẫu nhiên OK; số ca True =", trues, ", False =", falses)
assert trues > 50 and falses > 50

test ngẫu nhiên OK; số ca True = 454 , False = 2546


Test 3: tính chất — kết quả không đổi khi xáo trộn thứ tự `numbers`; không sửa input; và nếu ta thêm vào một số là tiền tố **thực sự** của một số đã có (cắt bớt ký tự cuối) thì kết quả chắc chắn là `False`.

In [16]:
random.seed(91)
for _ in range(500):
    n = random.randint(2, 20)
    pool = {"".join(random.choice("0123456789") for _ in range(random.randint(2, 12))) for _ in range(n)}
    numbers = list(pool)
    if len(numbers) < 2:
        continue
    snapshot = list(numbers)
    base = Solution().phonePrefix(numbers)
    assert numbers == snapshot
    shuffled = list(numbers)
    random.shuffle(shuffled)
    assert Solution().phonePrefix(shuffled) == base
    victim = random.choice(numbers)
    extra = victim[:-1]                      # tiền tố thực sự, độ dài >= 1 vì len(victim) >= 2
    if extra and extra not in pool and len(numbers) + 1 <= 50:
        assert Solution().phonePrefix(numbers + [extra]) is False
print("tính chất (xáo trộn, không sửa input, thêm tiền tố => False): OK")

tính chất (xáo trộn, không sửa input, thêm tiền tố => False): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `in` thay vì `startswith`

In [17]:
def wrong_in(numbers):
    for i in range(len(numbers)):
        for j in range(len(numbers)):
            if i != j and numbers[i] in numbers[j]:      # SAI: "nằm trong", không phải "bắt đầu bằng"
                return False
    return True

numbers = ["12", "312"]
print("sai :", wrong_in(numbers))
print("đúng:", Solution().phonePrefix(numbers))
assert wrong_in(numbers) != Solution().phonePrefix(numbers)

sai : False
đúng: True


Vì sao sai: `"12"` nằm **giữa** `"312"` chứ không ở đầu, nên không phải tiền tố; code sai trả `False` thay vì `True`. **Sửa:** `startswith`.

### Lỗi 2: so cặp kề **mà quên sort**

In [18]:
def wrong_no_sort(numbers):
    for i in range(len(numbers) - 1):
        if numbers[i + 1].startswith(numbers[i]):        # SAI: chưa sort nên tiền tố có thể không kề nhau
            return False
    return True

numbers = ["1", "2", "12"]
print("sai :", wrong_no_sort(numbers))
print("đúng:", Solution().phonePrefix(numbers))
assert wrong_no_sort(numbers) != Solution().phonePrefix(numbers)

sai : True
đúng: False


Vì sao sai: trong thứ tự gốc `"1"` và `"12"` bị `"2"` chen giữa nên không kề nhau; code sai trả `True` thay vì `False`. **Sửa:** sort trước khi so cặp kề.

### Lỗi 3: sắp xếp theo **giá trị số** (`key=int`)

In [19]:
def wrong_sort_by_int(numbers):
    s = sorted(numbers, key=int)                         # SAI: "001" -> 1, mất số 0 đầu và đổi thứ tự
    for i in range(len(s) - 1):
        if s[i + 1].startswith(s[i]):
            return False
    return True

numbers = ["001", "007", "15", "00153"]
print("sai :", wrong_sort_by_int(numbers), "| thứ tự theo int:", sorted(numbers, key=int))
print("đúng:", Solution().phonePrefix(numbers))
assert wrong_sort_by_int(numbers) != Solution().phonePrefix(numbers)

sai : True | thứ tự theo int: ['001', '007', '15', '00153']
đúng: False


Vì sao sai: theo giá trị số, `"15"` (15) đứng ngay trước `"00153"` (153) chứ không phải `"001"` (1); cặp kề thành `("15","00153")` không phải tiền tố, và cặp thật `("001","00153")` bị tách ra. **Sửa:** sort **chuỗi** (mặc định), không chuyển sang `int`.

### Lỗi 4: chỉ kiểm tra một chiều của cặp

In [20]:
def wrong_one_direction(numbers):
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if numbers[j].startswith(numbers[i]):        # SAI: chỉ xét "số sau bắt đầu bằng số trước"
                return False
    return True

numbers = ["00153", "001"]
print("sai :", wrong_one_direction(numbers))
print("đúng:", Solution().phonePrefix(numbers))
assert wrong_one_direction(numbers) != Solution().phonePrefix(numbers)

sai : True
đúng: False


Vì sao sai: số **ngắn** (tiền tố) có thể đứng **sau** số dài trong mảng; chỉ kiểm tra `j > i` bỏ sót trường hợp này. **Sửa:** kiểm tra cả hai chiều (hai vòng lặp `i != j`) hoặc dùng sort.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Một chuỗi có là **tiền tố** của chuỗi khác không" -> nghĩ ngay tới **sort từ điển** hoặc **Trie**.
- Sort chuỗi: tiền tố luôn đứng **ngay trước** một chuỗi bắt đầu bằng nó -> chỉ cần so cặp kề.
- Dữ liệu là **chuỗi chữ số**, đừng chuyển sang `int` (mất số `0` đầu, đổi thứ tự).
- Phân biệt `startswith` (đầu chuỗi) với `in` (bất kỳ chỗ nào).
- Nhớ xử lý hai chuỗi **bằng nhau**: tiền tố "không thực sự" — hãy đọc đề xem tính hay không (ở đây quy ước: tính).
- Bài họ hàng: `208. Implement Trie (Prefix Tree)`, `1268. Search Suggestions System`, `14. Longest Common Prefix`.

## 15. Biến thể của bài

1. **Trả về cặp vi phạm đầu tiên** hoặc **đếm** số cặp (tiền tố, số dài hơn).
2. **Hậu tố (suffix)** thay vì tiền tố: đảo ngược mỗi chuỗi rồi dùng cách cũ.
3. **Mảng rất lớn** (`10^5` chuỗi): dùng Trie hoặc sort; bỏ Cách 1, 3.
4. **Phone List kinh điển (UVa / Kattis "Phone List")**: cùng ý tưởng, nhiều test case.
5. **Tìm tiền tố chung dài nhất** của cả mảng (`14. Longest Common Prefix`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So mọi cặp | thử `startswith` cho mọi cặp có thứ tự | `O(n² · m)` | `O(1)` | oracle, `n` nhỏ |
| 2. Sort + cặp kề | tiền tố đứng sát số dài hơn | `O(n · m · log n)` | `O(n · m)` | **ngắn gọn, nên nộp** |
| 3. `set` + tiền tố | kiểm tra mọi tiền tố có trong tập | `O(n · m²)` | `O(n · m)` | dễ hiểu, `m` nhỏ |
| 4. Trie | cây tiền tố, đi và đánh dấu | `O(S)` | `O(S)` | tối ưu, mẫu cho bài tiền tố |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `phonePrefix(["91", "911", "9"])` bằng gì?

**Bài 2 (tính tay):** sắp xếp `["5", "55", "54", "6"]` theo thứ tự từ điển và cho biết có tiền tố hay không.

**Bài 3 (điền TODO):** hoàn thành `is_prefix(a, b)` trả về `True` nếu `a` là tiền tố **thực sự** của `b` (`a != b`).

**Bài 4 (tự viết lại):** viết `count_prefix_pairs(numbers)` đếm số cặp `(a, b)` (có thứ tự, `a != b` theo vị trí) mà `a` là tiền tố của `b`.

**Bài 5 (mini challenge):** viết `has_suffix_pair(numbers)` trả về `True` nếu có một số là **hậu tố** của số khác.

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().phonePrefix(["91", "911", "9"]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def is_prefix(a, b):
    # TODO: trả về True nếu b bắt đầu bằng a và a khác b
    return False

False


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** tiền tố = "bắt đầu bằng"; kiểm tra `"911".startswith("91")`.

**Mức 2 (vừa):** Bài 3: `return b.startswith(a) and a != b`. Bài 4: hai vòng lặp `i != j`, đếm khi `numbers[j].startswith(numbers[i])`. Bài 5: `startswith` đổi thành `endswith`.

**Mức 3 (gần đáp án):** Bài 1: `"9"` là tiền tố của `"91"` -> `False`. Bài 2: sắp xếp từ điển được `["5", "54", "55", "6"]`; `"5"` là tiền tố của `"54"` (kề nhau) -> có tiền tố.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
assert Solution().phonePrefix(["91", "911", "9"]) is False

# Bài 2
assert sorted(["5", "55", "54", "6"]) == ["5", "54", "55", "6"]
assert Solution().phonePrefix(["5", "55", "54", "6"]) is False

# Bài 3
def is_prefix(a, b):
    return b.startswith(a) and a != b
assert is_prefix("00", "001") is True
assert is_prefix("001", "00") is False
assert is_prefix("12", "12") is False
assert is_prefix("2", "12") is False

# Bài 4
def count_prefix_pairs(numbers):
    c = 0
    for i in range(len(numbers)):
        for j in range(len(numbers)):
            if i != j and numbers[j].startswith(numbers[i]):
                c += 1
    return c
assert count_prefix_pairs(["001", "007", "15", "00153"]) == 1
assert count_prefix_pairs(["1", "12", "123"]) == 3          # (1,12) (1,123) (12,123)
assert count_prefix_pairs(["1", "2"]) == 0

# Bài 5
def has_suffix_pair(numbers):
    for i in range(len(numbers)):
        for j in range(len(numbers)):
            if i != j and numbers[j].endswith(numbers[i]):
                return True
    return False
assert has_suffix_pair(["15", "315"]) is True
assert has_suffix_pair(["12", "21"]) is False
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "có chuỗi nào là tiền tố của chuỗi khác" (sort + cặp kề):**

```python
s = sorted(numbers)            # sort CHUỖI, không key=int
for i in range(len(s) - 1):
    if s[i + 1].startswith(s[i]):
        return False
return True
```

**Checklist trước khi submit:**

- [ ] Dùng `startswith` (đầu chuỗi), không dùng `in`?
- [ ] Đã sort **trước** khi so cặp kề, và sort theo **chuỗi** (không `key=int`)?
- [ ] Nếu so mọi cặp thì xét **cả hai chiều** `i != j`?
- [ ] Đã thử `n = 2`, tiền tố một ký tự, chuỗi toàn `0`, chuỗi dài `50`?
- [ ] Đã đọc đề về chuỗi **trùng hệt** (bài Premium, cần đối chiếu)?
- [ ] Chữ ký đúng `phonePrefix(self, numbers: List[str]) -> bool`?